# Tracking singola traiettoria, caso N=2 omodino

Notebook che riprende il setup di `main_SLURM.py` per il caso `N=2` omodino, ma salva gli stati di poche traiettorie e calcola osservabili **non mediate** tempo per tempo.

Il sistema qui e fissato a:
- `eta_1 = 1`
- `eta_2 = 0`
- `phi_1 = 0`
- `phi_2 = 0`

Le traiettorie simulate sono `10`.
Gli osservabili sono organizzati in **un DataFrame separato per ogni traiettoria**.
Se lo stato finale non ha fidelity finale abbastanza vicina a `1`, viene classificato nel bin separato `undetermined`.

Oltre alle fidelity nella base computazionale e di Bell, il notebook calcola anche le 4 fidelity nella base `+/-`: `|++\rangle, |+-\rangle, |-+\rangle, |--\rangle`.


In [ ]:
%matplotlib inline

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd()
CODE_DIR = ROOT / "Codes" if (ROOT / "Codes").exists() else ROOT
if str(CODE_DIR) not in sys.path:
    sys.path.append(str(CODE_DIR))

from single_trajectory_tracking_n2 import (
    BELL_FIDELITY_COLUMNS,
    COMPUTATIONAL_FIDELITY_COLUMNS,
    PLUS_MINUS_FIDELITY_COLUMNS,
    TrackingConfig,
    get_trajectory_dataframes,
    plot_concurrence,
    plot_final_state_histogram,
    plot_fidelity_grid,
    plot_spin_squeezing,
    run_tracking_experiment,
    save_figures,
    save_tracking_results,
)

plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["font.size"] = 12


In [ ]:
# Sistema fissato: non facciamo scan di eta o di fasi.
ETA_1 = 1.0
ETA_2 = 0.0
PHI_1 = 0.0
PHI_2 = 0.0

output_root = CODE_DIR / "Graphs" / "Jz_2_Homodyne_N2_single_trajectory"

config = TrackingConfig(
    state="plusplus",
    theta=np.pi / 2,
    phi_state=0.0,
    gamma=1.0,
    phi1=PHI_1,
    phi2=PHI_2,
    eta_1=ETA_1,
    eta_2=ETA_2,
    ntraj=10,
    t_end=5.0,
    dt=0.001,
    num_cpus=1,
    seed=12345,
    out_root=output_root,
)

config


In [ ]:
results = run_tracking_experiment(config)
out_dir = save_tracking_results(results)
trajectory_dfs = get_trajectory_dataframes(results)

print(f"Output salvato in: {out_dir}")
print(f"Shape stati = {results.states.shape}  # (ntraj, nsteps, 4, 4)")
print(f"Numero dataframe separati = {len(trajectory_dfs)}")
print(f"Traiettorie disponibili = {results.trajectory_ids}")

results.trajectory_frame(0).head()


In [ ]:
# Label finale per traiettoria.
# best_match_label = stato con fidelity finale massima tra gli 8 target
# steady_state_label = stesso stato solo se la fidelity finale e circa 1, altrimenti 'undetermined'
results.final_state_summary[[
    "trajectory",
    "best_match_label",
    "steady_state_label",
    "steady_state_fidelity",
    "steady_state_close_to_one",
]]


In [ ]:
for traj in results.trajectory_ids:
    df = results.trajectory_frame(traj)
    print(f"traj {traj + 1}: shape = {df.shape}")

results.trajectory_frame(1).head()


In [ ]:
fig_spin, _ = plot_spin_squeezing(results)
fig_conc, _ = plot_concurrence(results)

save_figures(
    {
        "SpinSqueezing_Trajectories": fig_spin,
        "Concurrence_Trajectories": fig_conc,
    },
    out_dir,
)

plt.show()


In [ ]:
fig_bell, _ = plot_fidelity_grid(
    results,
    BELL_FIDELITY_COLUMNS,
    "Bell fidelities per traiettoria",
)

fig_pm, _ = plot_fidelity_grid(
    results,
    PLUS_MINUS_FIDELITY_COLUMNS,
    "Fidelity nella base +/- per traiettoria",
)

fig_comp, _ = plot_fidelity_grid(
    results,
    COMPUTATIONAL_FIDELITY_COLUMNS,
    "Fidelity nella base computazionale per traiettoria",
)

save_figures(
    {
        "BellFidelities_Trajectories": fig_bell,
        "PlusMinusFidelities_Trajectories": fig_pm,
        "ComputationalFidelities_Trajectories": fig_comp,
    },
    out_dir,
)

plt.show()


In [ ]:
# Istogramma finale dei final state.
fig_hist, _ = plot_final_state_histogram(results)

save_figures(
    {
        "FinalStateHistogram": fig_hist,
    },
    out_dir,
)

plt.show()
